# Experiment 6: Model Registry and Version Management

### AIM
Implement model registration and lifecycle management by maintaining model versions across staging, validation, and production stages. Track model metadata, preprocessing dependencies, evaluation metrics, dataset lineage, and deployment readiness, and perform model comparison, promotion, rollback, and traceability throughout the ML lifecycle.

---
### Workflow Architecture:
1. **Model Registration**: Register candidate models (`LogisticRegression` V1, `LinearSVC` V2) in MLflow Model Registry under `News_Category_Classifier`.
2. **Preprocessing Dependency Lineage**: Link `models/preprocessor_pipeline.pkl` directly to registry model artifacts.
3. **Automated Lifecycle Management (Champion vs. Challenger)**:
   - Move newly registered models to `Staging`.
   - Identify top Challenger in `Staging` using weighted F1-Score.
   - Conduct automated Face-Off against current `Production` Champion.
   - Automatically promote superior Challenger $	o$ `Production` and archive previous Champion $	o$ `Archived`.
4. **Model Rollback Capability**: Programmatically revert to a previous version if required.
5. **Deployment Traceability Audit Report**: Export comprehensive audit log `artifacts/production_model_report.json`.
6. **Real-Time Production Inference Service**: Connect to MLflow Model Registry, load the active Production model, and classify incoming news headlines.


In [ ]:
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path("..").resolve() if Path(".").resolve().name == "notebooks" else Path(".").resolve()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print(f"Working Directory set to: {PROJECT_ROOT}")

## 1. Connect to MLflow Model Registry Client

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient
from src.config import MLFLOW_TRACKING_URI, MODEL_REGISTRY_NAME

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()
print(f"MLflow Registry URI: {MLFLOW_TRACKING_URI}")
print(f"Target Registered Model: '{MODEL_REGISTRY_NAME}'")

## 2. Inspect Registered Model Versions & Current Stages

In [ ]:
import pandas as pd
from IPython.display import display

versions = client.search_model_versions(f"name='{MODEL_REGISTRY_NAME}'")
v_records = []
for v in versions:
    v_records.append({
        "Version": v.version,
        "Stage": v.current_stage,
        "Run ID": v.run_id,
        "Status": v.status,
        "Source URI": v.source
    })

display(pd.DataFrame(v_records).sort_values("Version"))

## 3. Run Automated Champion-Challenger Lifecycle Management

In [ ]:
from src.registry_lifecycle_exp6 import manage_lifecycle_champion_challenger

# Automates stage discovery, staging face-off, promotion, and archival
manage_lifecycle_champion_challenger(primary_metric="f1_weighted")

## 4. View Production Deployment Traceability Report

In [ ]:
import json
from src.registry_lifecycle_exp6 import generate_production_registry_report

report = generate_production_registry_report()
print(json.dumps(report, indent=4))

## 5. Live Production Inference from Model Registry

In [ ]:
from src.inference_service import NewsInferenceService

# Initializes service by dynamically pulling the active 'Production' model from registry
service = NewsInferenceService()

sample_headlines = [
    ("Astronomers Detect Radio Signals From Deep Space Exoplanet", "New astrophysical observations challenge current cosmic theories."),
    ("Stock Market Rallies After Central Bank Cuts Benchmark Rates", "Investors celebrate easing monetary policy across major sectors."),
    ("New Electric Vehicle With 600-Mile Range Revealed At Auto Expo", "Automakers showcase breakthrough solid-state battery technology."),
    ("Heavy Rains Trigger Flash Flooding In Coastal Towns", "Emergency teams rescue stranded residents as rivers overflow.")
]

print("="*80)
print("REAL-TIME PRODUCTION CLASSIFICATION RESULTS")
print("="*80)
for head, desc in sample_headlines:
    pred = service.predict(head, desc)
    print(f"Headline:  {pred['headline']}")
    print(f"Category:  -> [ {pred['predicted_category']} ] (Model V{pred['production_model_version']} | Latency: {pred['inference_latency_ms']} ms)\n")


## 6. Model Rollback Demonstration
If a production issue arises with a deployed version, you can execute an instant rollback:
```python
from src.registry_lifecycle_exp6 import rollback_model_version

# Rollback to Version 1 if Version 2 experiences post-deployment drift
rollback_model_version(target_version=1)
```
